# Lab 01: Send Your First Amazon Bedrock Request

**Day 1 - Session 1**

Goal: Build and verify a minimal request to Amazon Bedrock through its OpenAI-compatible API.

> Open this notebook in Google Colab:
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/bedrock-companion/blob/main/day1/labs/01-first-bedrock-request/solution/01-first-bedrock-request.ipynb)

In [1]:
# Run this cell once to install the required package.
# If running locally with the course environment already activated, you can skip this.
!pip install "openai>=2.37.0" --quiet

In [2]:
import getpass
import re
from types import SimpleNamespace

from openai import OpenAI

## Section 1: Configure the Bedrock endpoint

Amazon Bedrock exposes an OpenAI-compatible endpoint for each AWS Region. Enter the instructor-approved Region and short-term key when prompted; `getpass` prevents the key from appearing in notebook output.

In [ ]:
DEFAULT_MODEL_ID = "openai.gpt-oss-20b-1:0"
DEFAULT_PROMPT = "Please summarize the following customer support case for a retail return:\n\nCustomer: I received the wrong size for my shoes (Order #12345). I ordered a size 10 but received a size 8.\nAction required: Summarize the issue and suggest the next steps for the agent."

aws_region = input("Instructor-approved AWS Region: ").strip()
bedrock_api_key = getpass.getpass("Short-term Amazon Bedrock API key: ").strip()

if not aws_region or not re.fullmatch(r"[a-z]{2}(?:-gov)?-[a-z]+-\d", aws_region):
    raise ValueError("Enter a valid AWS Region, such as us-east-1")
if not bedrock_api_key:
    raise ValueError("Amazon Bedrock API key must not be empty")

base_url = f"https://bedrock-runtime.{aws_region}.amazonaws.com/openai/v1"
client = OpenAI(api_key=bedrock_api_key, base_url=base_url)
print(f"Configured Amazon Bedrock in {aws_region}; key was not displayed.")

## Section 2: Implement the request function

Keep request logic separate from configuration by accepting an injected client. The function must reject blank input, make exactly one Chat Completions call, remove any leading reasoning block, and reject empty output.

In [4]:
def ask_bedrock(client, prompt, model_id=DEFAULT_MODEL_ID):
    """Send one user prompt and return a non-empty answer."""
    if not prompt.strip():
        raise ValueError("prompt must not be empty")

    response = client.chat.completions.create(
        model=model_id,
        messages=[{"role": "user", "content": prompt}],
    )
    content = response.choices[0].message.content
    answer = re.sub(r"<reasoning>.*?</reasoning>", "", content, flags=re.DOTALL).strip()
    if not answer:
        raise RuntimeError("Amazon Bedrock returned an empty text response")
    return answer

## Section 3: Run an offline acceptance check

This fake client records the request without using credentials or model tokens. Passing this check confirms the request shape and response cleanup before the live call.

In [10]:
class FakeCompletions:
    def __init__(self):
        self.calls = []

    def create(self, **kwargs):
        self.calls.append(kwargs)
        message = SimpleNamespace(content="<reasoning>private work</reasoning>  Test answer.  ")
        return SimpleNamespace(choices=[SimpleNamespace(message=message)])


fake_completions = FakeCompletions()
fake_client = SimpleNamespace(
    chat=SimpleNamespace(completions=fake_completions),
)
answer = ask_bedrock(fake_client, "What is Amazon Bedrock?", "test-model")

assert answer == "Test answer."
assert fake_completions.calls == [{
    "model": "test-model",
    "messages": [{"role": "user", "content": "What is Amazon Bedrock?"}],
}]
print("OFFLINE_CHECK_OK")

OFFLINE_CHECK_OK


## Section 4: Send one live request

The response wording is nondeterministic. Success means that the call returns non-empty text and the notebook prints the marker below.

In [18]:
live_answer = ask_bedrock(client, DEFAULT_PROMPT)
print(live_answer)
print(f"\nMODEL_RESPONSE_OK model={DEFAULT_MODEL_ID}")

**Amazon Bedrock** is AWS’s managed foundation‑model service that lets you build, deploy, and scale generative‑AI applications without having to own or maintain the underlying models.  Three standout features that make it especially useful for developers are:

| Feature | What it does | How it speeds up generative‑AI development |
|---------|--------------|-------------------------------------------|
| **1. One‑API access to a multi‑vendor model catalogue** | Bedrock lets you invoke models from several leading providers (Anthropic Claude, AI21 Labs, Stability AI, Amazon Titan, etc.) through a single, consistent REST/SDK API. | *You don’t need to juggle different auth flows or SDKs.*  You can pick the model that best matches your use case—e.g. Claude for nuanced dialogue, Stability AI for image generation—right from the same endpoint.  Switching or combining models becomes a matter of changing the payload, not rewiring your stack. |
| **2. Serverless, fully‑managed inference with auto‑s

## Cleanup

Restart the Colab runtime or Jupyter kernel now to clear the short-term key from memory. Do not save executed output that could contain sensitive error details.

In [19]:
print("Lab complete.")
print("Takeaway: An injected client keeps Bedrock request logic testable while endpoint and credential configuration remain separate.")

Lab complete.
Takeaway: An injected client keeps Bedrock request logic testable while endpoint and credential configuration remain separate.
